# Figure 3d — Axodendritic synapse counts across sensory neuron subtypes

Total number of axodendritic synapses made onto spinal cord neurons by the full central
projection of each reconstructed sensory neuron.

**Method.** Query CAVE `synapses_version3` for synapses where the sensory neuron is
**presynaptic**, drop autapses, and keep synapses with a cleft `size` >= 20 voxels
(Methods, *Quantification of axoaxonic and axodendritic synapses*).

**Statistics.** Kruskal-Wallis followed by Dunn post hoc (Holm correction).

Requires a CAVE token; see [`common/credentials.py`](../../common/credentials.py).


In [ ]:
import matplotlib as mpl

# Figure style used for all Figure 3 panels.
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Helvetica", "Arial", "DejaVu Sans"]
mpl.rcParams["svg.fonttype"] = "none"   # keep text editable in Illustrator
mpl.rcParams["font.size"] = 5


In [ ]:
# The 79 reconstructed sensory neurons, grouped by subtype
# (10 per subtype except C-Cold, n = 9).
SENSORY_NEURONS = {
    "c-ltmr": [
        720575940886931959, 720575940891163574, 720575940922958792,
        720575940918770481, 720575940893408011, 720575940880372359,
        720575940918990151, 720575940902671774, 720575940893122730,
        720575940914869232,
    ],
    "mrgd": [
        720575940915684544, 720575940921986112, 720575940920833705,
        720575940915967584, 720575940901960430, 720575940864403212,
        720575940915217983, 720575940927208416, 720575940941089388,
        720575940899256770,
    ],
    "ad-ltmr": [
        720575940902363320, 720575940879973250, 720575940909767128,
        720575940893392149, 720575940909235164, 720575940887704904,
        720575940909767384, 720575940913511793, 720575940908252144,
        720575940914044656,
    ],
    "ab-ltmr": [
        720575940882531294, 720575940893873415, 720575940881926778,
        720575940932756628, 720575940905014742, 720575940871491897,
        720575940892205986, 720575940884949941, 720575940898835143,
        720575940899788943,
    ],
    "sst": [
        720575940882155492, 720575940905081398, 720575940883866908,
        720575940918775345, 720575940882061434, 720575940920062581,
        720575940891751079, 720575940884108031, 720575940911489471,
        720575940927216096,
    ],
    "ad-htmr": [
        720575940903122154, 720575940884031660, 720575940884051238,
        720575940897347842, 720575940847552365, 720575940917989973,
        720575940922765354, 720575940882082080, 720575940884977867,
        720575940902680734,
    ],
    "chtmr-p": [
        720575940918288693, 720575940905365705, 720575940862564127,
        720575940916233731, 720575940881918801, 720575940910952353,
        720575940924479298, 720575940874972385, 720575940892182782,
        720575940914066622,
    ],
    "c-cold": [
        720575940872856946, 720575940883734520, 720575940890741967,
        720575940899184211, 720575940902618322, 720575940927175960,
        720575940927275084, 720575940878947380, 720575940900244450,
    ],
}


In [ ]:
# Internal subtype key -> published label
SUBTYPE_LABELS = {
    "c-ltmr":  "C-LTMR",
    "mrgd":    "C-HTMR/Heat (MrgprA3,MrgprD,MrgprB4)",
    "ad-ltmr": "Aδ-LTMR",
    "ab-ltmr": "Aβ-LTMR (single collateral)",
    "sst":     "C-HTMR/Heat (SST)",
    "ad-htmr": "Aδ-HTMR (single collateral)",
    "chtmr-p": "Peptidergic C-Nociceptors",
    "c-cold":  "C-Cold",
}

PALETTE = {
    "C-LTMR":                               "#BCEBCB",
    "C-HTMR/Heat (MrgprA3,MrgprD,MrgprB4)": "#b23aee",
    "Aδ-LTMR":                              "#00A86B",
    "Aβ-LTMR (single collateral)":          "#556B2F",
    "C-HTMR/Heat (SST)":                    "#ffbbff",
    "Aδ-HTMR (single collateral)":          "#ff4040",
    "Peptidergic C-Nociceptors":            "#ffee00",
    "C-Cold":                               "#00bfff",
}


In [ ]:
import caveclient

# Pin the materialization so the published root IDs and counts stay reproducible.
# 669 is the permanent release version. Set to None to query the latest instead.
MATERIALIZATION = 669

client = caveclient.CAVEclient("wclee_mouse_spinalcord_cltmr")
if MATERIALIZATION is not None:
    client.version = MATERIALIZATION

SYNAPSE_TABLE = "synapses_version3"
MIN_CLEFT_SIZE = 20   # voxels


In [ ]:
import numpy as np


def resolve_current_ids(client, root_ids):
    """Map the published root IDs onto their current equivalents.

    A CAVE root ID changes every time the neuron is edited, so IDs recorded when
    the figures were made go stale as proofreading continues. Any ID that is no
    longer current is replaced by the ChunkedGraph's suggested successor, and the
    substitution is reported. Without this, a stale ID silently returns zero
    synapses and the subtype mean drops.
    """
    ids = np.asarray([int(r) for r in root_ids], dtype=np.uint64)
    is_latest = client.chunkedgraph.is_latest_roots(ids)

    mapping, changed = {}, 0
    for rid, ok in zip(root_ids, is_latest):
        if ok:
            mapping[rid] = rid
            continue
        suggested = np.atleast_1d(client.chunkedgraph.suggest_latest_roots(int(rid)))
        mapping[rid] = int(suggested[0])
        changed += 1
        print(f"  stale: {rid} -> {mapping[rid]}")

    print(f"{changed} of {len(root_ids)} published root IDs were updated "
          f"for materialization {client.version}.")
    return mapping


In [ ]:
import pandas as pd

current = resolve_current_ids(
    client, [r for ids in SENSORY_NEURONS.values() for r in ids]
)

rows = []
for subtype, root_ids in SENSORY_NEURONS.items():
    for root_id in root_ids:
        syn = client.materialize.query_table(
            SYNAPSE_TABLE,
            filter_equal_dict={"pre_pt_root_id": current[root_id]},
        )
        if syn is not None and not syn.empty:
            syn = syn[
                (syn["pre_pt_root_id"] != syn["post_pt_root_id"])
                & (syn["size"] >= MIN_CLEFT_SIZE)
            ]
            n = len(syn)
        else:
            n = 0
        rows.append({"root_id": root_id, "Neuron Type": subtype,
                     "Synapse Count": n})

counts = pd.DataFrame(rows)
counts["Neuron Label"] = counts["Neuron Type"].map(SUBTYPE_LABELS)
print(counts.groupby("Neuron Label")["Synapse Count"]
            .agg(n="size", mean="mean", sd="std", median="median"))


In [ ]:
import numpy as np
import pandas as pd
import scikit_posthocs as sp
from scipy.stats import kruskal


def kruskal_dunn(df, value_col, group_col="Neuron Label"):
    """Kruskal-Wallis across subtypes, then Dunn post hoc with Holm correction."""
    groups = [g[value_col].to_numpy(float) for _, g in df.groupby(group_col)]
    H, p = kruskal(*groups)
    print(f"Kruskal-Wallis: H = {H:.4g}, p = {p:.4g}")

    dunn = sp.posthoc_dunn(df, val_col=value_col, group_col=group_col,
                           p_adjust="holm")
    return dunn


def bar_with_points(df, value_col, ylabel, order, figsize, yticks=None,
                    out_svg=None, seed=0):
    """Bar of subtype means with individual neurons overlaid as jittered points.

    Deliberately plain matplotlib: seaborn's barplot error-bar argument changed
    name across 0.11 / 0.12 (`ci=` -> `errorbar=`), so a seaborn version would
    break depending on which environment the notebook is run in.
    """
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=figsize)
    x = np.arange(len(order))

    means = [df.loc[df["Neuron Label"] == g, value_col].mean() for g in order]
    ax.bar(x, means, width=0.8, color=[PALETTE[g] for g in order],
           edgecolor="none", zorder=1)

    rng = np.random.default_rng(seed)
    for i, g in enumerate(order):
        vals = df.loc[df["Neuron Label"] == g, value_col].to_numpy(float)
        ax.scatter(i + rng.uniform(-0.2, 0.2, len(vals)), vals,
                   s=4, color="black", alpha=0.4, linewidths=0, zorder=3)

    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=-50, ha="left", fontsize=4)
    ax.set_xlim(-0.6, len(order) - 0.4)
    if yticks is not None:
        ax.set_yticks(yticks)
    ax.set_ylabel(ylabel, fontsize=5)
    ax.tick_params(axis="y", labelsize=5)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)
    fig.tight_layout()

    if out_svg:
        fig.savefig(out_svg, dpi=600, bbox_inches="tight")
    plt.show()


In [ ]:
dunn = kruskal_dunn(counts, "Synapse Count")
dunn


In [ ]:
# Subtypes ordered by descending mean, as published.
order = (counts.groupby("Neuron Label")["Synapse Count"].mean()
               .sort_values(ascending=False).index.tolist())

bar_with_points(
    counts,
    value_col="Synapse Count",
    ylabel="Synapse Count",
    order=order,
    figsize=(1.2, 0.8),
    yticks=[1000, 2000, 3000, 4000],
    out_svg="figure3d_axodendritic_synapse_counts.svg",
)
